In [1]:
"""
Makine öğrenmesi veri ön işleme pratikleri

Amaç:
    1. Eksik veri tespiti, çıkartılması ve uygun değerler ile doldurma
    2. IQR yöntemiyle sayısal sütunlardaki aykırı değerleri tespit etmek
    3. Kategorik verileri label encoding ve one-hot encoding ile dönüştür
    4. Veriyi train, validasyon ve test kümelerine ayır
    5. Sayısal özelliklere standardization ve normalization uygula

Kurulum:
pip install pandas scikit-learn
pip install -r requirements.txt
"""

'\nMakine öğrenmesi veri ön işleme pratikleri\n\nAmaç:\n    1. Eksik veri tespiti, çıkartılması ve uygun değerler ile doldurma\n    2. IQR yöntemiyle sayısal sütunlardaki aykırı değerleri tespit etmek\n    3. Kategorik verileri label encoding ve one-hot encoding ile dönüştür\n    4. Veriyi train, validasyon ve test kümelerine ayır\n    5. Sayısal özelliklere standardization ve normalization uygula\n\nKurulum:\npip install pandas scikit-learn\npip install -r requirements.txt\n'

In [2]:
# 1. gerekkli kütüphanelerin içeriye aktarılması
import pandas as pd

from sklearn.model_selection import train_test_split # eğitim ve test veri seti oluşturur
from sklearn.preprocessing import LabelEncoder, StandardScaler, MinMaxScaler 

In [3]:
# 2. veri setinin yüklenmesi
df = pd.read_csv("musteri_verisi_ml_pratik.csv")

In [4]:
print(df.head())
print(df.info())

    yas     maas     sehir         egitim  deneyim_yili satin_aldi
0  25.0  32000.0    Ankara         Lisans           1.0       Evet
1  31.0  41000.0  Istanbul  Yuksek Lisans           4.0      Hayir
2  28.0      NaN     Izmir         Lisans           2.0  Beklemede
3  45.0  67000.0    Ankara        Doktora          15.0      Hayir
4   NaN  52000.0     Bursa         Lisans           7.0       Evet
<class 'pandas.core.frame.DataFrame'>
RangeIndex: 12 entries, 0 to 11
Data columns (total 6 columns):
 #   Column        Non-Null Count  Dtype  
---  ------        --------------  -----  
 0   yas           11 non-null     float64
 1   maas          11 non-null     float64
 2   sehir         12 non-null     object 
 3   egitim        11 non-null     object 
 4   deneyim_yili  11 non-null     float64
 5   satin_aldi    12 non-null     object 
dtypes: float64(3), object(3)
memory usage: 708.0+ bytes
None


In [5]:
# 3. eksik veri analizi
print(df.isnull().sum())

yas             1
maas            1
sehir           0
egitim          1
deneyim_yili    1
satin_aldi      0
dtype: int64


In [6]:
df_dropna = df.dropna() # eksik veri çıkart
print(f"Eksik veriler çıktıktan sonra: \n{df_dropna}")

Eksik veriler çıktıktan sonra: 
     yas     maas     sehir         egitim  deneyim_yili satin_aldi
0   25.0  32000.0    Ankara         Lisans           1.0       Evet
1   31.0  41000.0  Istanbul  Yuksek Lisans           4.0      Hayir
3   45.0  67000.0    Ankara        Doktora          15.0      Hayir
5   36.0  48000.0  Istanbul      On Lisans           6.0  Beklemede
7   41.0  61000.0    Ankara  Yuksek Lisans          11.0      Hayir
8   33.0  45000.0     Bursa         Lisans           5.0  Beklemede
10  38.0  56000.0     Izmir  Yuksek Lisans           9.0       Evet
11  30.0  43000.0    Ankara         Lisans           3.0  Beklemede


In [7]:
df_filled = df.copy()

In [8]:
sayisal_sutunlar = ["yas", "maas", "deneyim_yili"]

In [9]:
# sayısal sütunları medyan ile doldur
for sutun in sayisal_sutunlar:
    medydan_degeri = df_filled[sutun].median()
    df_filled[sutun] = df_filled[sutun].fillna(medydan_degeri)

# kategorik sütunları en sık tekrar eden değer ile doldur
df_filled["egitim"] = df_filled["egitim"].fillna(df_filled["egitim"].mode()[0])

print(f"Eksik veriler doldurulduktan sonra: \n{df_filled}")

Eksik veriler doldurulduktan sonra: 
     yas     maas     sehir         egitim  deneyim_yili satin_aldi
0   25.0  32000.0    Ankara         Lisans           1.0       Evet
1   31.0  41000.0  Istanbul  Yuksek Lisans           4.0      Hayir
2   28.0  45000.0     Izmir         Lisans           2.0  Beklemede
3   45.0  67000.0    Ankara        Doktora          15.0      Hayir
4   31.0  52000.0     Bursa         Lisans           7.0       Evet
5   36.0  48000.0  Istanbul      On Lisans           6.0  Beklemede
6   29.0  39000.0     Izmir         Lisans           5.0       Evet
7   41.0  61000.0    Ankara  Yuksek Lisans          11.0      Hayir
8   33.0  45000.0     Bursa         Lisans           5.0  Beklemede
9   27.0  35000.0  Istanbul         Lisans           2.0      Hayir
10  38.0  56000.0     Izmir  Yuksek Lisans           9.0       Evet
11  30.0  43000.0    Ankara         Lisans           3.0  Beklemede


In [10]:
# 4. IQR yöntemiyle aykırı değerleri tespit etme

aykiri_deger_maskesi = pd.Series(False, index = df_filled.index)

In [12]:
for sutun in sayisal_sutunlar:

    q1 = df_filled[sutun].quantile(0.25)
    q3 = df_filled[sutun].quantile(0.75)

    iqr = q3 - q1

    alt_sinir = q1 - 1.5 * iqr
    ust_sinir = q3 + 1.5 * iqr

    sutun_maskesi = (
        (df_filled[sutun] < alt_sinir) | (df_filled[sutun] > ust_sinir)
    )

    aykiri_deger_maskesi = aykiri_deger_maskesi | sutun_maskesi

    print(f"Aykırı değer sayısı: {sutun_maskesi.sum()}")

    if sutun_maskesi.any():
        print(f"Aykırı değerler: \n{df_filled.loc[sutun_maskesi, sutun]}")

print(f"En az bir aykırı değer içeren satırlar \n{df_filled.loc[aykiri_deger_maskesi]}")

# aykırı değer içeren satırları veri setinden çıkartalım
df_clean = df_filled.loc[~aykiri_deger_maskesi].copy()
df_clean.reset_index(drop=True, inplace=True)

print(f"Aykırı değerler çıktıktan sonra \n{df_clean}")

Aykırı değer sayısı: 0
Aykırı değer sayısı: 0
Aykırı değer sayısı: 1
Aykırı değerler: 
3    15.0
Name: deneyim_yili, dtype: float64
En az bir aykırı değer içeren satırlar 
    yas     maas   sehir   egitim  deneyim_yili satin_aldi
3  45.0  67000.0  Ankara  Doktora          15.0      Hayir
Aykırı değerler çıktıktan sonra 
     yas     maas     sehir         egitim  deneyim_yili satin_aldi
0   25.0  32000.0    Ankara         Lisans           1.0       Evet
1   31.0  41000.0  Istanbul  Yuksek Lisans           4.0      Hayir
2   28.0  45000.0     Izmir         Lisans           2.0  Beklemede
3   31.0  52000.0     Bursa         Lisans           7.0       Evet
4   36.0  48000.0  Istanbul      On Lisans           6.0  Beklemede
5   29.0  39000.0     Izmir         Lisans           5.0       Evet
6   41.0  61000.0    Ankara  Yuksek Lisans          11.0      Hayir
7   33.0  45000.0     Bursa         Lisans           5.0  Beklemede
8   27.0  35000.0  Istanbul         Lisans           2.0      Hay

In [13]:
# 5. label encoding ve one-hot encoding 

label_encoder = LabelEncoder()

# hedef değişkeni sayısal hale getir
y = label_encoder.fit_transform(df_clean["satin_aldi"])

print(f"Hedef değişken sınıfları: \n {label_encoder.classes_}")
print(y)

Hedef değişken sınıfları: 
 ['Beklemede' 'Evet' 'Hayir']
[1 2 0 1 0 1 2 0 2 1 0]


In [14]:
# hedef sütunu veri setinden çıkart
X = df_clean.drop(columns=["satin_aldi"])

X = pd.get_dummies(X, columns=["egitim"], drop_first=True, dtype=int)

print(f"Kategorik dönüşüm sonrası özellikler:  \n{X}")

Kategorik dönüşüm sonrası özellikler:  
     yas     maas     sehir  deneyim_yili  egitim_On Lisans  \
0   25.0  32000.0    Ankara           1.0                 0   
1   31.0  41000.0  Istanbul           4.0                 0   
2   28.0  45000.0     Izmir           2.0                 0   
3   31.0  52000.0     Bursa           7.0                 0   
4   36.0  48000.0  Istanbul           6.0                 1   
5   29.0  39000.0     Izmir           5.0                 0   
6   41.0  61000.0    Ankara          11.0                 0   
7   33.0  45000.0     Bursa           5.0                 0   
8   27.0  35000.0  Istanbul           2.0                 0   
9   38.0  56000.0     Izmir           9.0                 0   
10  30.0  43000.0    Ankara           3.0                 0   

    egitim_Yuksek Lisans  
0                      0  
1                      1  
2                      0  
3                      0  
4                      0  
5                      0  
6             

In [15]:
# 6. Veriyi train validasyon ve test kümelerine ayır
X_train_val, X_test, y_train_val, y_test = train_test_split(X, y, test_size=0.2, random_state=42, stratify=y) # val = %80, test = %20

X_train, X_val, y_train, y_val = train_test_split(X_train_val, y_train_val, test_size=0.4, random_state=42, stratify=y_train_val)

print(f"X_train: {X_train.shape}")
print(f"X_val: {X_val.shape}")
print(f"X_test: {X_test.shape}")


X_train: (4, 6)
X_val: (4, 6)
X_test: (3, 6)


In [16]:
# 7. sayısal özelliklerde standardization 

standard_scaler = StandardScaler()

X_train_standard = X_train.copy()
X_val_standard = X_val.copy()
X_test_standard = X_test.copy()

# ölçekleyiciyi yalnızca eğitim verisi üzerinde öğretiyoruz
X_train_standard[sayisal_sutunlar] = (
    standard_scaler.fit_transform(
        X_train[sayisal_sutunlar]
    )
)

# validasyon ve test verilerinde yalnızca transform uygula
X_val_standard[sayisal_sutunlar] = (
    standard_scaler.transform(
        X_val[sayisal_sutunlar]
    )
)

X_test_standard[sayisal_sutunlar] = (
    standard_scaler.transform(
        X_test[sayisal_sutunlar]
    )
)


In [17]:
print(f"X_train_standard: \n{X_train_standard}")

X_train_standard: 
        yas      maas     sehir  deneyim_yili  egitim_On Lisans  \
1 -0.973329 -1.489269  Istanbul      -1.38675                 0   
9  1.297771  1.218493     Izmir       1.38675                 0   
3 -0.973329  0.496423     Bursa       0.27735                 0   
4  0.648886 -0.225647  Istanbul      -0.27735                 1   

   egitim_Yuksek Lisans  
1                     1  
9                     1  
3                     0  
4                     0  


In [18]:
# 8. normalizasyon

minmax_scaler = MinMaxScaler()

X_train_normalized = X_train.copy()
X_val_normalized = X_val.copy()
X_test_normalized = X_test.copy()

# ölçekleyiciyi yalnızca eğitim verisi üzerinde öğretiyoruz
X_train_normalized[sayisal_sutunlar] = (
    minmax_scaler.fit_transform(
        X_train[sayisal_sutunlar]
    )
)

# validasyon ve test verilerinde yalnızca transform uygula
X_val_normalized[sayisal_sutunlar] = (
    minmax_scaler.transform(
        X_val[sayisal_sutunlar]
    )
)

X_test_normalized[sayisal_sutunlar] = (
    minmax_scaler.transform(
        X_test[sayisal_sutunlar]
    )
)

print(f"X_train_normalized: \n{X_train_normalized}")

X_train_normalized: 
        yas      maas     sehir  deneyim_yili  egitim_On Lisans  \
1  0.000000  0.000000  Istanbul           0.0                 0   
9  1.000000  1.000000     Izmir           1.0                 0   
3  0.000000  0.733333     Bursa           0.6                 0   
4  0.714286  0.466667  Istanbul           0.4                 1   

   egitim_Yuksek Lisans  
1                     1  
9                     1  
3                     0  
4                     0  
